In [1]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer, DistilBertTokenizerFast
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])

# Only keep Arabic, Telugu and Korean examples
df_train = df_train[df_train['lang'].isin(['ar', 'te', 'ko'])]


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/andreasmelbye/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/Users/andreasmelbye/miniconda3/envs/nlp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
[(row['context'][row['answer_start']:row['answer_start']+len(row['answer'])], row['answer']) for _, row in df_train.iterrows()]

[('France', 'France'),
 ('Wilhelm Röntgen', 'Wilhelm Röntgen'),
 ('2004', '2004'),
 ('British Broadcasting Corporation (BBC)',
  'British Broadcasting Corporation (BBC)'),
 ('Jerusalem', 'Jerusalem'),
 ('Andromeda', 'Andromeda'),
 ('Smith–Putnam wind turbine', 'Smith–Putnam wind turbine'),
 ('Louis, Dauphin', 'Louis, Dauphin'),
 ('CBS', 'CBS'),
 ('Chandragupta Maurya', 'Chandragupta Maurya'),
 ('appointed "Reichsführer-SS" by Hitler',
  'appointed "Reichsführer-SS" by Hitler'),
 ('Silicon', 'Silicon'),
 ('1945', '1945'),
 ('Subway', 'Subway'),
 ('22 November 2005', '22 November 2005'),
 ('London', 'London'),
 ('USSR', 'USSR'),
 ('', 'no'),
 ('one-quarter', 'one-quarter'),
 ('Vertigo', 'Vertigo'),
 ('7th place', '7th place'),
 ('Charles Darwin', 'Charles Darwin'),
 ('Islamic State', 'Islamic State'),
 ('Revelation', 'Revelation'),
 ('UuU', 'UuU'),
 ('Sydney', 'Sydney'),
 ('"MUD1"', '"MUD1"'),
 ('2,932, excluding military personnel serving in the archipelago and their dependents. A 2012 

In [3]:
def create_bio_tags_and_indices(context, answer_start, answer):
    """
    Creates BIO tags and indices for a given context and answer.
    """
    # Use a regex to tokenize words and punctuation
    tokens = re.findall(r'\b\w+\b|.', context)
    
    # Calculate character indices
    token_data = []
    current_index = 0
    for token in tokens:
        start_index = context.find(token, current_index)
        end_index = start_index + len(token)
        token_data.append({
            'word': token,
            'start': start_index,
            'end': end_index,
            'tag': 'O'  # Default tag
        })
        current_index = end_index

    # Assign BIO tags
    if not pd.isna(answer_start) and not pd.isna(answer):
        answer_end = answer_start + len(answer)
        is_inside_answer = False
        for i, token in enumerate(token_data):
            word_start = token['start']
            word_end = token['end']

            # Check for overlap with the answer's character span
            if max(word_start, answer_start) < min(word_end, answer_end):
                if not is_inside_answer:
                    token_data[i]['tag'] = 'B'
                    is_inside_answer = True
                else:
                    token_data[i]['tag'] = 'I'
            else:
                is_inside_answer = False

    return token_data

In [4]:
index = 4

# Process the specific row from the DataFrame
context = df_train.iloc[index]['context']
answer_start = df_train.iloc[index]['answer_start']
answer = df_train.iloc[index]['answer']

# Get the tagged tokens
tagged_tokens = create_bio_tags_and_indices(context, answer_start, answer)

# Print the output in the user's desired format
for token in tagged_tokens:
    print(f"{token['word']:15} -> {token['tag']:>2} -> ({token['start']},{token['end']})")

Palestine       ->  O -> (0,9)
                ->  O -> (9,10)
(               ->  O -> (10,11)
                ->  O -> (11,12)
'               ->  O -> (12,13)
)               ->  O -> (13,14)
,               ->  O -> (14,15)
                ->  O -> (15,16)
officially      ->  O -> (16,26)
                ->  O -> (26,27)
the             ->  O -> (27,30)
                ->  O -> (30,31)
State           ->  O -> (31,36)
                ->  O -> (36,37)
of              ->  O -> (37,39)
                ->  O -> (39,40)
Palestine       ->  O -> (40,49)
                ->  O -> (49,50)
(               ->  O -> (50,51)
                ->  O -> (51,52)
'               ->  O -> (52,53)
)               ->  O -> (53,54)
,               ->  O -> (54,55)
                ->  O -> (55,56)
is              ->  O -> (56,58)
                ->  O -> (58,59)
a               ->  O -> (59,60)
                ->  O -> (60,61)
"               ->  O -> (61,62)
de              ->  O -> (62,64)
             